# Projet Data - Analyse internationale de la satisfaction client

Pipeline de data wrangling et de profilage des enquêtes des cinq filiales.

In [46]:
from pathlib import Path
import importlib
import sys

import pandas as pd

BASE_DIR = Path.cwd()
if BASE_DIR.name != "pipeline_v1":
    BASE_DIR = Path(".").resolve()

sys.path.insert(0, str(BASE_DIR / "modules"))

import data_pipeline
import normalization
import standardization
import rag

importlib.reload(data_pipeline)
importlib.reload(normalization)
importlib.reload(standardization)
importlib.reload(rag)

from data_pipeline import (
    build_missing_report,
    build_structure_report,
    calculate_correlations,
    calculate_outlier_rates,
    compare_missing_values,
    correct_data_quality,
    filter_low_correlation,
    impute_numeric_missing,
    load_datasets,
)
from normalization import export_normalized_outputs, normalize_all
from standardization import standardize_all
from rag import answer_with_rag

DATA_DIR = BASE_DIR / "BDD_raw"
STANDARD_DIR = BASE_DIR / "BDD_standard_format"
NORMALIZED_DIR = BASE_DIR / "BDD_normalized_format"
LOW_CORRELATION_THRESHOLD = 0.2

# Sprint 1 : 24/09/2026 - 07/10/2026



## 2.1.1 : Analyse de la structure des datasets

Les fichiers sont lus depuis `BDD_raw`, puis leur structure est comparée.

In [47]:
datasets = load_datasets(DATA_DIR)
structure_report = build_structure_report(datasets)

print(f"Datasets charges : {', '.join(datasets)}")
display(structure_report[[
    "country",
    "rows",
    "columns",
    "numeric_columns",
    "categorical_columns",
    "duplicate_rows",
    "total_missing",
    "missing_rate_global_pct",
    "target_column",
]])

Datasets charges : Allemagne, Canada, Espagne, France, Maroc


,country,rows,columns,numeric_columns,categorical_columns,duplicate_rows,total_missing,missing_rate_global_pct,target_column
0,Allemagne,200,14,7,7,0,50,1.79,None
1,Canada,200,14,7,7,0,50,1.79,overall_satisfaction
2,Espagne,200,14,2,12,0,50,1.79,None
3,France,200,14,7,7,0,50,1.79,satisfaction_globale
4,Maroc,200,10,2,8,0,10,0.50,None


### Profilage des valeurs manquantes

In [48]:
missing_report = build_missing_report(datasets)
display(missing_report.sort_values("missing_rate_pct", ascending=False))

,country,column,dtype,missing_count,row_count,missing_rate_pct
40,Espagne,servicio_cliente,object,13,200,6.5
12,Allemagne,kundenservice,float64,13,200,6.5
53,France,etat_colis,float64,12,200,6.0
52,France,relation_livreur,float64,12,200,6.0
41,Espagne,satisfaccion_global,object,12,200,6.0
...,...,...,...,...,...,...
59,Maroc,code_produit,object,0,200,0.0
61,Maroc,fragile,object,0,200,0.0
62,Maroc,poids_kg,float64,0,200,0.0
63,Maroc,prix_livraison_eur,float64,0,200,0.0


# Sprint 2 : 08/10/2026 - 21/10/2026



## 2.2.1 : Filtrage des variables faiblement corrélées

Le filtrage s'applique uniquement aux colonnes numériques. Les variables dont `|r| < 0.2` sont retirées par pays.

In [49]:
correlations = calculate_correlations(datasets)
filtered_datasets, removed_columns = filter_low_correlation(
    datasets,
    correlations,
    threshold=LOW_CORRELATION_THRESHOLD,
)

print(f"Variables retirees : {len(removed_columns)}")
display(correlations)
display(removed_columns)

Variables retirees : 3


,country,target,variable,pearson_r,abs_r,n_valid
0,Canada,overall_satisfaction,poids_kg,0.012462,0.012462,189
1,Canada,overall_satisfaction,prix_livraison_eur,0.028442,0.028442,189
2,Canada,overall_satisfaction,delivery_time,0.142686,0.142686,181
3,Canada,overall_satisfaction,delivery_person,0.100912,0.100912,180
4,Canada,overall_satisfaction,parcel_condition,0.187453,0.187453,182
5,Canada,overall_satisfaction,customer_service,0.290693,0.290693,178
6,France,satisfaction_globale,poids_kg,-0.008638,0.008638,189
7,France,satisfaction_globale,prix_livraison_eur,0.075347,0.075347,189
8,France,satisfaction_globale,delai_livraison,0.300197,0.300197,184
9,France,satisfaction_globale,relation_livreur,0.278077,0.278077,177


,country,target,variable,pearson_r,abs_r,n_valid,threshold,justification
2,Canada,overall_satisfaction,delivery_time,0.142686,0.142686,181,0.2,Pearson |r| < 0.2: faible correlation avec le ...
3,Canada,overall_satisfaction,delivery_person,0.100912,0.100912,180,0.2,Pearson |r| < 0.2: faible correlation avec le ...
4,Canada,overall_satisfaction,parcel_condition,0.187453,0.187453,182,0.2,Pearson |r| < 0.2: faible correlation avec le ...


## 2.3.1 : Traitement des données manquantes

Taux de valeurs manquantes supérieur à 5 % : médiane. Sinon : moyenne.

In [50]:
imputed_datasets, imputation_log = impute_numeric_missing(filtered_datasets)
missing_comparison = compare_missing_values(filtered_datasets, imputed_datasets)

display(imputation_log)
display(missing_comparison)

,country,column,missing_rate_before_pct,missing_before,method,imputation_value,missing_after
0,Allemagne,poids_kg,0.0,0,no_missing,NaN,0
1,Allemagne,prix_livraison_eur,0.0,0,no_missing,NaN,0
2,Allemagne,lieferzeit,5.5,11,median,4.000000,0
3,Allemagne,zusteller,4.5,9,mean,3.633508,0
4,Allemagne,paket_zustand,3.5,7,mean,3.533679,0
5,Allemagne,kundenservice,6.5,13,median,4.000000,0
6,Allemagne,gesamtzufriedenheit,5.0,10,mean,3.831579,0
7,Canada,poids_kg,0.0,0,no_missing,NaN,0
8,Canada,prix_livraison_eur,0.0,0,no_missing,NaN,0
9,Canada,customer_service,5.5,11,median,7.000000,0


,country,missing_before,missing_after,missing_removed
0,Allemagne,50,0,50
1,Canada,22,0,22
2,Espagne,50,50,0
3,France,50,0,50
4,Maroc,10,10,0


## 2.4.1 : Calcul du taux de valeurs aberrantes

Le taux est calculé pour chaque variable numérique avec la méthode de l'écart interquartile (IQR).

In [51]:
outlier_report = calculate_outlier_rates(imputed_datasets)
display(outlier_report.sort_values("outlier_rate_pct", ascending=False))

,country,column,valid_count,outlier_count,outlier_rate_pct,q1,q3,iqr,lower_bound,upper_bound
13,France,poids_kg,200,18,9.0,1.1850,5.7150,4.5300,-5.61000,12.51000
0,Allemagne,poids_kg,200,17,8.5,1.2800,5.4850,4.2050,-5.02750,11.79250
7,Canada,poids_kg,200,16,8.0,1.2775,5.4600,4.1825,-4.99625,11.73375
20,Maroc,poids_kg,200,16,8.0,1.2600,5.4700,4.2100,-5.05500,11.78500
11,Espagne,poids_kg,200,15,7.5,1.2275,5.3475,4.1200,-4.95250,11.52750
4,Allemagne,paket_zustand,200,15,7.5,3.0000,4.0000,1.0000,1.50000,5.50000
16,France,relation_livreur,200,14,7.0,3.0000,4.0000,1.0000,1.50000,5.50000
18,France,service_client,200,13,6.5,3.0000,4.0000,1.0000,1.50000,5.50000
5,Allemagne,kundenservice,200,11,5.5,3.0000,4.0000,1.0000,1.50000,5.50000
3,Allemagne,zusteller,200,9,4.5,3.0000,4.0000,1.0000,1.50000,5.50000


## 2.4.2 : Correction des valeurs aberrantes et des catégories manquantes

Les catégories manquantes sont remplacées par leur mode. Les valeurs numériques aberrantes sont plafonnées aux bornes IQR.

In [52]:
corrected_datasets, correction_log = correct_data_quality(
    imputed_datasets,
    outlier_report,
)

display(correction_log)

,country,column,issue,count_before,count_corrected,method,value
0,Allemagne,poids_kg,numeric_outlier,17,17,clip_iqr_bounds,"[-5.027499999999999, 11.792499999999999]"
1,Allemagne,prix_livraison_eur,numeric_outlier,4,4,clip_iqr_bounds,"[-3.098749999999999, 26.87125]"
2,Allemagne,zusteller,numeric_outlier,9,9,clip_iqr_bounds,"[1.5, 5.5]"
3,Allemagne,paket_zustand,numeric_outlier,15,15,clip_iqr_bounds,"[1.5, 5.5]"
4,Allemagne,kundenservice,numeric_outlier,11,11,clip_iqr_bounds,"[1.5, 5.5]"
5,Canada,poids_kg,numeric_outlier,16,16,clip_iqr_bounds,"[-4.99625, 11.73375]"
6,Canada,prix_livraison_eur,numeric_outlier,6,6,clip_iqr_bounds,"[-2.4387499999999998, 24.151249999999997]"
7,Canada,customer_service,numeric_outlier,3,3,clip_iqr_bounds,"[1.5, 13.5]"
8,Canada,overall_satisfaction,numeric_outlier,5,5,clip_iqr_bounds,"[3.0, 11.0]"
9,Espagne,plazo_entrega,missing_categorical,9,9,mode,Alto


## Fin du data profiling

Les résultats finaux sont affichés ci-dessous. Aucun fichier de rapport n'est exporté.

In [53]:
print("Fin du data profiling")
print("\nStructure des datasets")
display(structure_report)
print("\nValeurs manquantes")
display(missing_report)
print("\nVariables retirees")
display(removed_columns)
print("\nJournal d'imputation")
display(imputation_log)
print("\nComparaison avant / apres")
display(missing_comparison)
print("\nValeurs aberrantes")
display(outlier_report)
print("\nJournal de correction")
display(correction_log)

Fin du data profiling

Structure des datasets


,country,rows,columns,numeric_columns,categorical_columns,duplicate_rows,total_missing,missing_rate_global_pct,target_column,dtypes
0,Allemagne,200,14,7,7,0,50,1.79,None,umfrage_id: object; umfrage_datum: datetime64[...
1,Canada,200,14,7,7,0,50,1.79,overall_satisfaction,survey_id: object; survey_date: datetime64[ns]...
2,Espagne,200,14,2,12,0,50,1.79,None,id_encuesta: object; fecha_encuesta: object; p...
3,France,200,14,7,7,0,50,1.79,satisfaction_globale,id_enquete: object; date_enquete: datetime64[n...
4,Maroc,200,10,2,8,0,10,0.50,None,id_enquete: object; date_enquete: datetime64[n...



Valeurs manquantes


,country,column,dtype,missing_count,row_count,missing_rate_pct
0,Allemagne,umfrage_id,object,0,200,0.0
1,Allemagne,umfrage_datum,datetime64[ns],0,200,0.0
2,Allemagne,land,object,0,200,0.0
3,Allemagne,produkt_code,object,0,200,0.0
4,Allemagne,taille_colis,object,0,200,0.0
...,...,...,...,...,...,...
61,Maroc,fragile,object,0,200,0.0
62,Maroc,poids_kg,float64,0,200,0.0
63,Maroc,prix_livraison_eur,float64,0,200,0.0
64,Maroc,statut_colis,object,0,200,0.0



Variables retirees


...

,country,missing_before,missing_after,missing_removed
0,Allemagne,50,0,50
1,Canada,22,0,22
2,Espagne,50,50,0
3,France,50,0,50
4,Maroc,10,10,0



Valeurs aberrantes


,country,column,valid_count,outlier_count,outlier_rate_pct,q1,q3,iqr,lower_bound,upper_bound
0,Allemagne,poids_kg,200,17,8.5,1.2800,5.4850,4.2050,-5.02750,11.79250
1,Allemagne,prix_livraison_eur,200,4,2.0,8.1400,15.6325,7.4925,-3.09875,26.87125
2,Allemagne,lieferzeit,200,0,0.0,3.0000,5.0000,2.0000,0.00000,8.00000
3,Allemagne,zusteller,200,9,4.5,3.0000,4.0000,1.0000,1.50000,5.50000
4,Allemagne,paket_zustand,200,15,7.5,3.0000,4.0000,1.0000,1.50000,5.50000
5,Allemagne,kundenservice,200,11,5.5,3.0000,4.0000,1.0000,1.50000,5.50000
6,Allemagne,gesamtzufriedenheit,200,0,0.0,3.0000,5.0000,2.0000,0.00000,8.00000
7,Canada,poids_kg,200,16,8.0,1.2775,5.4600,4.1825,-4.99625,11.73375
8,Canada,prix_livraison_eur,200,6,3.0,7.5325,14.1800,6.6475,-2.43875,24.15125
9,Canada,customer_service,200,3,1.5,6.0000,9.0000,3.0000,1.50000,13.50000



Journal de correction


,country,column,issue,count_before,count_corrected,method,value
0,Allemagne,poids_kg,numeric_outlier,17,17,clip_iqr_bounds,"[-5.027499999999999, 11.792499999999999]"
1,Allemagne,prix_livraison_eur,numeric_outlier,4,4,clip_iqr_bounds,"[-3.098749999999999, 26.87125]"
2,Allemagne,zusteller,numeric_outlier,9,9,clip_iqr_bounds,"[1.5, 5.5]"
3,Allemagne,paket_zustand,numeric_outlier,15,15,clip_iqr_bounds,"[1.5, 5.5]"
4,Allemagne,kundenservice,numeric_outlier,11,11,clip_iqr_bounds,"[1.5, 5.5]"
5,Canada,poids_kg,numeric_outlier,16,16,clip_iqr_bounds,"[-4.99625, 11.73375]"
6,Canada,prix_livraison_eur,numeric_outlier,6,6,clip_iqr_bounds,"[-2.4387499999999998, 24.151249999999997]"
7,Canada,customer_service,numeric_outlier,3,3,clip_iqr_bounds,"[1.5, 13.5]"
8,Canada,overall_satisfaction,numeric_outlier,5,5,clip_iqr_bounds,"[3.0, 11.0]"
9,Espagne,plazo_entrega,missing_categorical,9,9,mode,Alto


# Sprint 3 : 22/10/2026 - 04/11/2026



## 3.1.1 : Standardisation vers Excel puis JSON

Les cinq datasets traités sont convertis vers un schéma commun, puis écrits dans `BDD_standard_format`.

In [54]:
standardized_datasets, standardization_report = standardize_all(
    corrected_datasets,
    STANDARD_DIR,
)

display(standardization_report)

for country, df in standardized_datasets.items():
    print(f"{country}: {df.shape[0]} lignes, {df.shape[1]} colonnes")

,country,rows,columns,missing_columns,excel_path,json_path,status
0,Allemagne,200,15,commentaire_global,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,success
1,Canada,200,15,"delai_livraison, relation_livreur, etat_colis,...",c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,success
2,Espagne,200,15,commentaire_global,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,success
3,France,200,15,commentaire_global,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,success
4,Maroc,200,15,"delai_livraison, relation_livreur, etat_colis,...",c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Pro...,success


Allemagne: 200 lignes, 15 colonnes
Canada: 200 lignes, 15 colonnes
Espagne: 200 lignes, 15 colonnes
France: 200 lignes, 15 colonnes
Maroc: 200 lignes, 15 colonnes


## 3.2.1 : Normalisation des notes sur 5

Les données standardisées et corrigées sont converties vers une échelle commune de satisfaction.

## 3.3.1 : Référentiel produit mondial

Les codes produits locaux sont associés à `STANDARD`, `EXPRESS` ou `PREMIUM`.

In [55]:
normalized_datasets, normalized_base, normalization_report = normalize_all(
    standardized_datasets,
)

assert len(normalized_base) == 1000
assert normalized_base["produit_global"].notna().all()
assert normalized_base["satisfaction_normalisee"].dropna().between(1, 5).all()
assert normalized_base["id_unique"].notna().all()
assert normalized_base["id_unique"].is_unique

normalization_outputs = export_normalized_outputs(
    normalized_datasets,
    normalized_base,
    NORMALIZED_DIR,
)

display(normalization_report)
display(normalized_base.head())

for name, path in normalization_outputs.items():
    print(f"{name}: {path}")

,pays,lignes,colonnes,scores_renseignes,produits_manquants
0,Allemagne,200,12,200,0
1,Canada,200,12,200,0
2,Espagne,200,12,200,0
3,France,200,12,200,0
4,Maroc,200,12,200,0


,id_unique,date_enquete,pays,code_produit,produit_global,taille_colis,fragile,poids_kg,prix_livraison_eur,statut_colis,satisfaction_normalisee,commentaire_global
0,DE-0001,2026-06-18 00:00:00,Allemagne,PAK_003,PREMIUM,Moyen,Non,4.5300,18.07,Livré,3.831579,NaN
1,DE-0002,2026-09-07 00:00:00,Allemagne,PAK_001,STANDARD,Petit,Non,1.2600,6.48,Livré,4.000000,NaN
2,DE-0003,2026-04-25 00:00:00,Allemagne,PAK_001,STANDARD,Petit,Non,0.7500,6.12,Livré,3.000000,NaN
3,DE-0004,2026-06-16 00:00:00,Allemagne,PAK_001,STANDARD,Grand,Non,11.7925,15.47,Retardé,5.000000,NaN
4,DE-0005,2026-01-16 00:00:00,Allemagne,PAK_001,STANDARD,Moyen,Oui,2.4200,10.09,Livré,3.831579,NaN


Allemagne_excel: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\allemagne_normalisee.xlsx
Allemagne_json: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\allemagne_normalisee.json
Canada_excel: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\canada_normalisee.xlsx
Canada_json: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\canada_normalisee.json
Espagne_excel: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\espagne_normalisee.xlsx
Espagne_json: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\espagne_normalisee.json
France_excel: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_fast_colis\pipeline_v1\BDD_normalized_format\france_normalisee.xlsx
France_json: c:\Users\Khyro\Desktop\PGE3\Data Wrangling\Projet_

# Sprint 4 : 05/11/2026 - 18/11/2026

## 4.1.1 : Interrogation des donnees avec un RAG

Le systeme retrouve les lignes les plus pertinentes dans la base normalisee, puis transmet uniquement ce contexte a Ollama. Le modele utilise `mistral:latest` par defaut. Ollama doit etre lance localement et le modele doit etre disponible.

In [56]:
rag_data = globals().get("normalized_base")
if rag_data is None:
    normalized_path = NORMALIZED_DIR / "fastcolis_base_normalisee.json"
    if not normalized_path.exists():
        raise FileNotFoundError(
            "Base normalisee introuvable. Executez d'abord les cellules du pipeline "
            "jusqu'a la normalisation."
        )
    rag_data = pd.read_json(normalized_path)

question_rag = input("Posez votre question sur les enquetes FastColis : ")
reponse_rag = answer_with_rag(
    rag_data,
    question_rag,
    model="mistral:latest",
    top_k=8,
 )
print(reponse_rag)

Le produit dont le score global est le plus élevé est le "COLIS_PREMIUM". Son ID unique est inconnu car il n'est pas fourni dans les lignes de données.
